In [ ]:
import os
import gc
import random
import warnings
import re
import math

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import pydicom

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from torchvision.models import (
    resnet50,
    ResNet50_Weights
)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / 1024**3,
            2
        ),
        "GB"
    )

In [ ]:
INPUT_ROOT = "/kaggle/input"

print(
    os.listdir(INPUT_ROOT)
)

In [ ]:
DATA_DIR = (
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

print(
    os.listdir(DATA_DIR)
)

In [ ]:
train = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "train.csv"
    )
)

train_series = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "train_series.csv"
    )
)

test = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "test.csv"
    )
)

test_series = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "test_series.csv"
    )
)

sample_submission = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "sample_submission.csv"
    )
)

print(
    "Train:",
    train.shape
)

print(
    "Train series:",
    train_series.shape
)

print(
    "Test:",
    test.shape
)

print(
    "Test series:",
    test_series.shape
)

print(
    "Sample submission:",
    sample_submission.shape
)

In [ ]:
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

NUM_CLASSES = len(TARGETS)

print(
    "Number of targets:",
    NUM_CLASSES
)

for i, target in enumerate(TARGETS):
    print(i, target)

In [ ]:
print(
    "Total studies:",
    len(train)
)

print(
    "\nNon-null labels:"
)

print(
    train[TARGETS].notna().sum()
)

In [ ]:
labeled_mask = (
    train[TARGETS]
    .notna()
    .all(axis=1)
)

unlabeled_mask = (
    train[TARGETS]
    .isna()
    .all(axis=1)
)

labeled_data = train[
    labeled_mask
].copy()

unlabeled_data = train[
    unlabeled_mask
].copy()

print(
    "Officially labeled studies:",
    len(labeled_data)
)

print(
    "Unlabeled studies:",
    len(unlabeled_data)
)

In [ ]:
missing_pattern = (
    train[TARGETS]
    .isna()
    .astype(int)
    .value_counts()
)

display(
    missing_pattern
)

In [ ]:
official_counts = (
    labeled_data[TARGETS]
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    official_counts.to_frame(
        "Positive Cases"
    )
)

In [ ]:
plt.figure(
    figsize=(14, 6)
)

official_counts.plot(
    kind="bar"
)

plt.title(
    "Officially Labeled MRI Studies"
)

plt.ylabel(
    "Positive Cases"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

In [ ]:
display(
    train_series.head()
)

In [ ]:
print(
    train_series[
        "Anatomical_Plane"
    ].value_counts()
)

In [ ]:
print(
    train_series[
        "Fluid_Sensitive"
    ].value_counts()
)

print()

print(
    train_series[
        "Fat_Suppression"
    ].value_counts()
)

In [ ]:
TRAIN_SERIES_DIR = os.path.join(
    DATA_DIR,
    "train_series"
)

TEST_SERIES_DIR = os.path.join(
    DATA_DIR,
    "test_series"
)

print(
    TRAIN_SERIES_DIR
)

print(
    TEST_SERIES_DIR
)

In [ ]:
def get_dicom_paths(
    study_id,
    series_id,
    base_dir
):

    series_path = os.path.join(
        base_dir,
        str(study_id),
        str(series_id)
    )

    if not os.path.exists(
        series_path
    ):
        return []

    paths = []

    for filename in os.listdir(
        series_path
    ):

        if filename.lower().endswith(
            ".dcm"
        ):

            paths.append(
                os.path.join(
                    series_path,
                    filename
                )
            )

    return paths

In [ ]:
def get_sorted_dicom_paths(
    study_id,
    series_id,
    base_dir
):

    paths = get_dicom_paths(
        study_id,
        series_id,
        base_dir
    )

    records = []

    for path in paths:

        try:

            dcm = pydicom.dcmread(
                path,
                stop_before_pixels=True
            )

            instance_number = getattr(
                dcm,
                "InstanceNumber",
                0
            )

            records.append(
                (
                    instance_number,
                    path
                )
            )

        except Exception:
            continue

    records.sort(
        key=lambda x: x[0]
    )

    return [
        path
        for _, path in records
    ]

In [ ]:
def read_dicom(
    path
):

    dcm = pydicom.dcmread(
        path
    )

    image = dcm.pixel_array.astype(
        np.float32
    )

    if getattr(
        dcm,
        "PhotometricInterpretation",
        ""
    ) == "MONOCHROME1":

        image = (
            image.max() - image
        )

    # Apply rescale slope/intercept
    slope = float(
        getattr(
            dcm,
            "RescaleSlope",
            1
        )
    )

    intercept = float(
        getattr(
            dcm,
            "RescaleIntercept",
            0
        )
    )

    image = (
        image * slope
        + intercept
    )

    # Percentile normalization
    low = np.percentile(
        image,
        1
    )

    high = np.percentile(
        image,
        99
    )

    if high > low:

        image = (
            image - low
        ) / (
            high - low
        )

    else:

        image = np.zeros_like(
            image
        )

    image = np.clip(
        image,
        0,
        1
    )

    return image

In [ ]:
def select_representative_slices(
    paths,
    num_slices=8
):

    if len(paths) == 0:
        return []

    if len(paths) <= num_slices:

        return paths

    indices = np.linspace(
        0,
        len(paths) - 1,
        num_slices
    ).astype(int)

    return [
        paths[i]
        for i in indices
    ]

In [ ]:
def select_series(
    study_id,
    series_df,
    max_series=3
):

    df = series_df[
        series_df[
            "StudyInstanceUID"
        ] == study_id
    ].copy()

    if len(df) == 0:
        return df

    # Priority:
    # 1. Fluid-sensitive
    # 2. Fat suppression
    # 3. Anatomical plane

    df["priority"] = (
        df["Fluid_Sensitive"].astype(int)
        * 2
        +
        df["Fat_Suppression"].astype(int)
    )

    df = df.sort_values(
        "priority",
        ascending=False
    )

    return df.head(
        max_series
    )

In [ ]:
sample_study = labeled_data.iloc[0][
    "StudyInstanceUID"
]

print(
    "Sample study:",
    sample_study
)

In [ ]:
sample_series = select_series(
    sample_study,
    train_series,
    max_series=3
)

display(
    sample_series
)

In [ ]:
series_id = sample_series.iloc[0][
    "SeriesInstanceUID"
]

paths = get_sorted_dicom_paths(
    sample_study,
    series_id,
    TRAIN_SERIES_DIR
)

print(
    "Number of slices:",
    len(paths)
)

In [ ]:
selected_paths = (
    select_representative_slices(
        paths,
        num_slices=8
    )
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 8)
)

for ax, path in zip(
    axes.ravel(),
    selected_paths
):

    image = read_dicom(
        path
    )

    ax.imshow(
        image,
        cmap="gray"
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
for i in range(
    min(10, len(train))
):

    print(
        "=" * 80
    )

    print(
        train.iloc[i]["Report"]
    )

In [ ]:
def normalize_text(
    text
):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Normalize accents
    import unicodedata

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(
            char
        )
    )

    # Replace line breaks
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
KEYWORDS = {

    "ACL": [
        "acl",
        "anterior cruciate",
        "ligament croise anterieur",
        "ligamento cruzado anterior",
        "vorderes kreuzband"
    ],

    "MCL": [
        "mcl",
        "medial collateral",
        "ligament collateral medial",
        "ligamento colateral medial",
        "innenband"
    ],

    "Medial Meniscus": [
        "medial meniscus",
        "menisque medial",
        "menisco medial",
        "medialmeniskus"
    ],

    "Lateral Meniscus": [
        "lateral meniscus",
        "menisque lateral",
        "menisco lateral",
        "lateralmeniskus"
    ],

    "Medial OA": [
        "medial osteoarthritis",
        "medial compartment osteoarthritis",
        "medial compartment",
        "osteoarthritis medial",
        "arthrose medial",
        "gonarthrose medial"
    ],

    "Lateral OA": [
        "lateral osteoarthritis",
        "lateral compartment osteoarthritis",
        "osteoarthritis lateral",
        "arthrose lateral"
    ],

    "PF OA": [
        "patellofemoral osteoarthritis",
        "patellofemoral arthrosis",
        "patellofemoral arthritis",
        "pf osteoarthritis",
        "patellofemoral"
    ],

    "Effusion": [
        "joint effusion",
        "effusion",
        "knee effusion",
        "epanchement",
        "derrame",
        "gelenkerguss"
    ],

    "Synovitis": [
        "synovitis",
        "synovial inflammation",
        "synoviale inflammation"
    ],

    "Baker's": [
        "baker cyst",
        "baker's cyst",
        "popliteal cyst",
        "kyste de baker",
        "quiste de baker",
        "bakerzyste"
    ],

    "Contusion": [
        "bone contusion",
        "bone bruise",
        "osseous contusion",
        "bone marrow edema",
        "bone oedema",
        "bone edema",
        "contusion osseuse"
    ],

    "Fracture": [
        "fracture",
        "fractura",
        "fracture osseuse",
        "fraktur"
    ]
}

In [ ]:
NEGATION_WORDS = [
    "no",
    "not",
    "without",
    "normal",
    "intact",
    "negative",
    "absent",
    "aucune",
    "aucun",
    "sans",
    "normal",
    "normale",
    "keine",
    "kein",
    "keinen",
    "keiner",
    "nein",
    "sin",
    "sano",
    "sana"
]

In [ ]:
def detect_finding(
    text,
    keywords,
    window=100
):

    text = normalize_text(
        text
    )

    for keyword in keywords:

        keyword = normalize_text(
            keyword
        )

        start = 0

        while True:

            position = text.find(
                keyword,
                start
            )

            if position == -1:
                break

            context_start = max(
                0,
                position - window
            )

            context_end = min(
                len(text),
                position + len(keyword) + window
            )

            context = text[
                context_start:context_end
            ]

            # Check negation words
            negative = any(
                re.search(
                    r"\b" +
                    re.escape(word) +
                    r"\b",
                    context
                )
                for word in NEGATION_WORDS
            )

            if not negative:

                return 1

            start = (
                position
                + len(keyword)
            )

    return 0

In [ ]:
def create_weak_labels(
    dataframe
):

    result = dataframe[
        [
            "StudyInstanceUID",
            "Report"
        ]
    ].copy()

    for target in TARGETS:

        keywords = KEYWORDS[
            target
        ]

        result[target] = (
            result["Report"]
            .apply(
                lambda x:
                detect_finding(
                    x,
                    keywords
                )
            )
        )

    return result

In [ ]:
weak_labels = create_weak_labels(
    train
)

display(
    weak_labels.head()
)

In [ ]:
weak_counts = (
    weak_labels[TARGETS]
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    weak_counts.to_frame(
        "Weak Positive Count"
    )
)

In [ ]:
comparison = pd.DataFrame(
    index=TARGETS
)

comparison[
    "Official Positive"
] = (
    labeled_data[TARGETS]
    .sum()
)

comparison[
    "Weak Positive"
] = (
    weak_labels[
        weak_labels[
            "StudyInstanceUID"
        ].isin(
            labeled_data[
                "StudyInstanceUID"
            ]
        )
    ][TARGETS]
    .sum()
)

display(
    comparison
)

In [ ]:
official = labeled_data[
    TARGETS
].copy()

weak_official = weak_labels[
    weak_labels[
        "StudyInstanceUID"
    ].isin(
        labeled_data[
            "StudyInstanceUID"
        ]
    )
].copy()

weak_official = (
    weak_official
    .set_index(
        "StudyInstanceUID"
    )
    .loc[
        labeled_data[
            "StudyInstanceUID"
        ]]
)

official.index = (
    labeled_data[
        "StudyInstanceUID"
    ]
)

agreement = (
    official[TARGETS]
    .values
    ==
    weak_official[TARGETS]
    .values
)

agreement_rate = (
    agreement.mean(
        axis=0
    )
)

agreement_table = pd.DataFrame({
    "Target": TARGETS,
    "Agreement": agreement_rate
})

display(
    agreement_table
)

In [ ]:
training_data = train[
    [
        "StudyInstanceUID",
        "Report"
    ]
].copy()

training_data = training_data.merge(
    weak_labels,
    on=[
        "StudyInstanceUID",
        "Report"
    ],
    how="left"
)

In [ ]:
official_map = labeled_data.set_index(
    "StudyInstanceUID"
)[TARGETS]

for target in TARGETS:

    training_data[target] = (
        training_data[target]
        .astype(float)
    )

    for study_id in official_map.index:

        official_value = (
            official_map.loc[
                study_id,
                target
            ]
        )

        training_data.loc[
            training_data[
                "StudyInstanceUID"
            ] == study_id,
            target
        ] = official_value

In [ ]:
print(
    training_data[
        TARGETS
    ].isna().sum()
)


In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        8
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

valid_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [ ]:
def prepare_image(
    image,
    transform
):

    image = (
        image * 255
    ).clip(
        0,
        255
    ).astype(
        np.uint8
    )

    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    return transform(
        image
    )

In [ ]:
class KneeDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        series_dataframe,
        base_dir,
        targets,
        transform=None,
        num_slices=8,
        max_series=2
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.series_df = (
            series_dataframe
        )

        self.base_dir = base_dir

        self.targets = targets

        self.transform = transform

        self.num_slices = (
            num_slices
        )

        self.max_series = (
            max_series
        )

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def load_images(
        self,
        study_id
    ):

        series_info = (
            select_series(
                study_id,
                self.series_df,
                self.max_series
            )
        )

        images = []

        for _, row in (
            series_info.iterrows()
        ):

            series_id = row[
                "SeriesInstanceUID"
            ]

            paths = (
                get_sorted_dicom_paths(
                    study_id,
                    series_id,
                    self.base_dir
                )
            )

            selected_paths = (
                select_representative_slices(
                    paths,
                    self.num_slices
                )
            )

            for path in selected_paths:

                try:

                    image = read_dicom(
                        path
                    )

                    if self.transform:

                        image = (
                            prepare_image(
                                image,
                                self.transform
                            )
                        )

                    images.append(
                        image
                    )

                except Exception:
                    continue

        return images

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        study_id = row[
            "StudyInstanceUID"
        ]

        images = self.load_images(
            study_id
        )

        if len(images) == 0:

            image = torch.zeros(
                3,
                IMAGE_SIZE,
                IMAGE_SIZE
            )

        else:

            image = random.choice(
                images
            )

        labels = torch.tensor(
            row[
                self.targets
            ].values.astype(
                np.float32
            ),
            dtype=torch.float32
        )

        return (
            image,
            labels
        )

In [ ]:
official_train, official_valid = (
    train_test_split(
        labeled_data,
        test_size=0.2,
        random_state=42
    )
)

print(
    "Official train:",
    len(official_train)
)

print(
    "Official validation:",
    len(official_valid)
)

In [ ]:
official_train_ids = set(
    official_train[
        "StudyInstanceUID"
    ]
)

official_valid_ids = set(
    official_valid[
        "StudyInstanceUID"
    ]
)

# Remove validation studies
# from pseudo-labelled training

pseudo_training = training_data[
    ~training_data[
        "StudyInstanceUID"
    ].isin(
        official_valid_ids
    )
].copy()

print(
    "Training studies:",
    len(pseudo_training)
)

In [ ]:
train_dataset = KneeDataset(
    dataframe=pseudo_training,
    series_dataframe=train_series,
    base_dir=TRAIN_SERIES_DIR,
    targets=TARGETS,
    transform=train_transform,
    num_slices=8,
    max_series=2
)

valid_dataset = KneeDataset(
    dataframe=official_valid,
    series_dataframe=train_series,
    base_dir=TRAIN_SERIES_DIR,
    targets=TARGETS,
    transform=valid_transform,
    num_slices=8,
    max_series=2
)

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Validation dataset:",
    len(valid_dataset)
)

In [ ]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "DataLoaders ready."
)

In [ ]:
images, labels = next(
    iter(train_loader)
)

print(
    "Images:",
    images.shape
)

print(
    "Labels:",
    labels.shape
)

In [ ]:
class KneeResNet(
    nn.Module
):

    def __init__(
        self,
        num_classes=12
    ):

        super().__init__()

        self.backbone = resnet50(
            weights=(
                ResNet50_Weights.DEFAULT
            )
        )

        features = (
            self.backbone.fc.in_features
        )

        self.backbone.fc = nn.Linear(
            features,
            num_classes
        )

    def forward(
        self,
        x
    ):

        return self.backbone(
            x
        )

In [ ]:
model = KneeResNet(
    num_classes=NUM_CLASSES
)

model = model.to(
    DEVICE
)

print(
    "Model ready."
)

In [ ]:
criterion = (
    nn.BCEWithLogitsLoss()
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1
    )
)

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    total_loss = 0

    progress = tqdm(
        loader,
        desc="Training"
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
        )

        progress.set_postfix(
            loss=round(
                loss.item(),
                4
            )
        )

    return (
        total_loss /
        len(loader)
    )

In [ ]:
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0

    all_predictions = []

    all_labels = []

    with torch.no_grad():

        for images, labels in tqdm(
            loader,
            desc="Validation"
        ):

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                labels
            )

            total_loss += (
                loss.item()
            )

            probabilities = (
                torch.sigmoid(
                    logits
                )
            )

            all_predictions.append(
                probabilities
                .cpu()
                .numpy()
            )

            all_labels.append(
                labels
                .cpu()
                .numpy()
            )

    predictions = np.concatenate(
        all_predictions,
        axis=0
    )

    labels = np.concatenate(
        all_labels,
        axis=0
    )

    auc_scores = []

    for i in range(
        NUM_CLASSES
    ):

        try:

            auc = roc_auc_score(
                labels[:, i],
                predictions[:, i]
            )

        except ValueError:

            auc = np.nan

        auc_scores.append(
            auc
        )

    mean_auc = np.nanmean(
        auc_scores
    )

    avg_loss = (
        total_loss /
        len(loader)
    )

    return (
        avg_loss,
        mean_auc,
        auc_scores,
        predictions,
        labels
    )

In [ ]:
EPOCHS = 3

best_auc = -np.inf

history = {
    "train_loss": [],
    "valid_loss": [],
    "valid_auc": []
}

for epoch in range(
    EPOCHS
):

    print(
        "\n"
        + "=" * 60
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print(
        "=" * 60
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    (
        valid_loss,
        valid_auc,
        auc_scores,
        predictions,
        labels
    ) = validate(
        model,
        valid_loader,
        criterion,
        DEVICE
    )

    scheduler.step(
        valid_auc
    )

    history[
        "train_loss"
    ].append(
        train_loss
    )

    history[
        "valid_loss"
    ].append(
        valid_loss
    )

    history[
        "valid_auc"
    ].append(
        valid_auc
    )

    print(
        f"Train Loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation Loss: "
        f"{valid_loss:.4f}"
    )

    print(
        f"Mean ROC-AUC: "
        f"{valid_auc:.4f}"
    )

    if valid_auc > best_auc:

        best_auc = valid_auc

        torch.save(
            model.state_dict(),
            "/kaggle/working/"
            "best_knee_model.pth"
        )

        print(
            "Best model saved."
        )

In [ ]:
auc_table = pd.DataFrame({

    "Abnormality": TARGETS,

    "ROC_AUC": auc_scores

})

display(
    auc_table.sort_values(
        "ROC_AUC",
        ascending=False
    )
)

In [ ]:
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history["train_loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    history["valid_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Training History"
)

plt.legend()

plt.show()

In [ ]:
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history["valid_auc"],
    marker="o"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "Validation ROC-AUC"
)

plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "/kaggle/working/"
        "best_knee_model.pth",
        map_location=DEVICE
    )
)

model.eval()

print(
    "Best model loaded."
)

In [ ]:
class KneeTestDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        series_dataframe,
        base_dir,
        transform=None,
        num_slices=8,
        max_series=2
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.series_df = (
            series_dataframe
        )

        self.base_dir = base_dir

        self.transform = transform

        self.num_slices = (
            num_slices
        )

        self.max_series = (
            max_series
        )

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[
            idx
        ]

        study_id = row[
            "StudyInstanceUID"
        ]

        series_info = select_series(
            study_id,
            self.series_df,
            self.max_series
        )

        images = []

        for _, series_row in (
            series_info.iterrows()
        ):

            series_id = (
                series_row[
                    "SeriesInstanceUID"
                ]
            )

            paths = (
                get_sorted_dicom_paths(
                    study_id,
                    series_id,
                    self.base_dir
                )
            )

            selected_paths = (
                select_representative_slices(
                    paths,
                    self.num_slices
                )
            )

            for path in selected_paths:

                try:

                    image = read_dicom(
                        path
                    )

                    image = prepare_image(
                        image,
                        self.transform
                    )

                    images.append(
                        image
                    )

                except Exception:
                    continue

        if len(images) == 0:

            image = torch.zeros(
                3,
                IMAGE_SIZE,
                IMAGE_SIZE
            )

        else:

            image = random.choice(
                images
            )

        return (
            image,
            study_id
        )

In [ ]:
test_dataset = KneeTestDataset(
    dataframe=test,
    series_dataframe=test_series,
    base_dir=TEST_SERIES_DIR,
    transform=valid_transform,
    num_slices=8,
    max_series=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Test studies:",
    len(test_dataset)
)

In [ ]:
model.eval()

test_predictions = []

test_ids = []

with torch.no_grad():

    for images, study_ids in tqdm(
        test_loader,
        desc="Predicting"
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            images
        )

        probabilities = (
            torch.sigmoid(
                logits
            )
        )

        test_predictions.append(
            probabilities
            .cpu()
            .numpy()
        )

        test_ids.extend(
            list(study_ids)
        )

In [ ]:
test_predictions = np.concatenate(
    test_predictions,
    axis=0
)

submission = pd.DataFrame(
    test_predictions,
    columns=TARGETS
)

submission.insert(
    0,
    "StudyInstanceUID",
    test_ids
)

display(
    submission.head()
)

In [ ]:
print(
    "Minimum prediction:",
    submission[TARGETS]
    .min()
    .min()
)

print(
    "Maximum prediction:",
    submission[TARGETS]
    .max()
    .max()
)

In [ ]:
print(
    "Sample columns:"
)

print(
    sample_submission.columns.tolist()
)

print(
    "\nOur columns:"
)

print(
    submission.columns.tolist()
)

In [ ]:
submission = submission[
    sample_submission.columns
]

In [ ]:
SUBMISSION_PATH = (
    "/kaggle/working/"
    "submission.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

print(
    "Submission saved:"
)

print(
    SUBMISSION_PATH
)

In [ ]:
submission_check = pd.read_csv(
    SUBMISSION_PATH
)

assert list(
    submission_check.columns
) == list(
    sample_submission.columns
)

assert len(
    submission_check
) == len(
    sample_submission
)

assert (
    submission_check[
        TARGETS
    ].isna()
    .sum()
    .sum()
    == 0
)

assert (
    submission_check[
        TARGETS
    ].values >= 0
).all()

assert (
    submission_check[
        TARGETS
    ].values <= 1
).all()

print(
    "================================"
)

print(
    "SUBMISSION VALIDATION PASSED"
)

print(
    "================================"
)

display(
    submission_check.head()
)

In [ ]:
# FINAL SUBMISSION

submission = submission[
    sample_submission.columns
].copy()

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("submission.csv created successfully!")
print(submission.shape)
display(submission.head())

In [ ]:
import os

path = "/kaggle/working/submission.csv"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("File size:", os.path.getsize(path), "bytes")
    print("Ready for Kaggle submission!")